In [1]:
import numpy as np
import matplotlib.pyplot as plt
import re
import math
import random

In [2]:
with open('some_words.txt' , 'r' , encoding='utf-8') as f:
    corpus = f.read()
    corpus = re.sub(r'[^a-z\s]', ' ', corpus.lower())
words = corpus.lower().split()
words[:10]

['alice',
 'was',
 'beginning',
 'to',
 'get',
 'very',
 'tired',
 'of',
 'sitting',
 'by']

In [4]:
vocab = sorted(list(set(words))) #-> unique words hai 
word_idx = {w:i for i , w in enumerate(vocab)}
idx_word = {i:w for i , w in enumerate(vocab)}

In [5]:
# creating training pairs of consecutive words
X_data = [word_idx[words[i]] for i in range(len(words) - 1)] # -> indexes ko nikaal rahe hai.
Y_data = [word_idx[words[i + 1]] for i in range(len(words) - 1)] # -> indexes main se 1 kam kar diya 

In [6]:
# own Value class:-
class value:
    def __init__(self ,data , _children = (), _op = '' , label = '' ):
        self.data =  data
        self.grad = 0.0
        self._backward = lambda : None
        self._prev = set(_children)
        self.label = label
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self , other):
        other = other if isinstance(other,value) else value(other)
        out = value(self.data+other.data , (self , other) , '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out

    def __pow__(self , other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * (self.data ** (other - 1))) * out.grad
        out._backward = _backward
        return out



    def __rmul__(self,other): # self * other
        return self * other

    def __radd__(self, other):
        return self + other

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __neg__(self): # -self
        return self * -1

    def __sub__(self, other): # self - other
        return self + (-other)


    def __mul__(self , other):
        other = other if isinstance(other,value) else value(other)
        out = value(self.data * other.data , (self , other) , '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def log(self):
        out = value(math.log(self.data), (self,), 'log')

        def _backward():
            self.grad += (1 / self.data) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        x = math.exp(self.data)
        out = value(x, (self,), 'exp')

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward

        return out


    def tanh(self):
        t = math.tanh(self.data)
        out = value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward

        return out


    def backward(self):
        self.grad = 1.0
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
            topo.append(v)
        build_topo(self)
        for node in reversed(topo):
            node._backward() 

In [7]:
class Neuron:
    def __init__(self , no_input):
        self.weights = [value(random.uniform(-1 ,1)) for _ in range(no_input)]
        self.bias = value(random.uniform(-1, 1))

    def __call__(self , x):
        # We wanted to multiply all inputs with their weights and then add bais to them.
        act = sum((wi * xi for wi, xi in zip(self.weights, x)), self.bias)
        out = act.tanh()
        return out         

    def parameters(self):
        return self.weights + [self.bias] 

class Layer:
    def __init__(self, no_input, no_neuron):
        self.neurons = [Neuron(no_input) for _ in range (no_neuron)] # -> number of inpput == number of neuron in next layer

    def __call__(self , x):
        out = [n(x) for n in self.neurons] # -> 
        return out[0] if len(out) == 1 else out
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

    
class MLP:
    def __init__(self, no_input , no_neuron):
        size = [no_input] + no_neuron
        self.layers = [Layer(size[i] , size[i+1]) for i in range(len(no_neuron))]

    def __call__(self ,x):
        for l in self.layers:
            x = l(x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
V = len(vocab)
def one_hot(idx , size):
    vec = [value(0.0) for _ in range(size)] # -> size ki value ke jitne 0s
    vec[idx] = value(1.0) # -> X_data ke index ki jangan par 1 laga de raha
    return vec

X_enc = [one_hot(x, V) for x in X_data]

In [10]:
model = MLP(V , [16 , V]) # -> input size = 2558 , 1 hidden layer of 16 units, output size = 2558

In [9]:
def softmax(l):
    exp_values = [x.exp() for x in l]

    total = sum(exp_values, value(0.0))

    prob = [x / total for x in exp_values]

    return prob

In [ ]:
ll = value(0.0)

for x, y in zip(X_enc, Y_data):

    logit = model(x)

    probs = softmax(logit)

    p = probs[y]

    ll += p.log()


nll = - ll / len(X_enc)

print(f"log_likelyhood = {ll.data}")
print(f"NLL = {nll.data}") 

# 

# Now using Pytorch
# note:(if you have much computation power then you can try this out because my laptop terribly hanging sorry :) guys😭)

In [11]:
import torch
import torch.nn as nn

In [23]:
class ContextMLP(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, block_size=3, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        # Flatten concatenated context embeddings: block_size * emb_dim
        self.fc1 = nn.Linear(block_size * emb_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x shape: (batch_size, block_size)
        h = self.emb(x).view(x.size(0), -1) # Flatten context embeddings
        h = self.act(self.fc1(h))
        logits = self.fc2(h)
        return logits

model = ContextMLP(vocab_size=V, emb_dim=64, block_size=3, hidden_dim=128)

In [22]:
# Context size of 3 words instead of 1
block_size = 3
X_data, Y_data = [], []

for i in range(len(words) - block_size):
    X_data.append([word_idx[w] for w in words[i:i+block_size]])
    Y_data.append(word_idx[words[i+block_size]])

X_data = torch.tensor(X_data) # Shape: (N, 3)
Y_data = torch.tensor(Y_data)

In [24]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

In [25]:
epochs = 200

for epoch in range(epochs):
    # 1. Forward pass (processes all samples at once)
    logits = model(X_data)
    loss = criterion(logits, Y_data)

    # 2. Backward pass
    optimizer.zero_grad()
    loss.backward()
    
    # 3. Update weights
    optimizer.step()

    # Print progress
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Loss (NLL): {loss.item():.4f}")

Epoch   1/200 | Loss (NLL): 7.8987
Epoch  20/200 | Loss (NLL): 4.0289
Epoch  40/200 | Loss (NLL): 2.1212
Epoch  60/200 | Loss (NLL): 1.2813
Epoch  80/200 | Loss (NLL): 0.7936
Epoch 100/200 | Loss (NLL): 0.5063
Epoch 120/200 | Loss (NLL): 0.3478
Epoch 140/200 | Loss (NLL): 0.2691
Epoch 160/200 | Loss (NLL): 0.2356
Epoch 180/200 | Loss (NLL): 0.2221
Epoch 200/200 | Loss (NLL): 0.2164


# We will see it's output in 'usingPytorch.ipynb' file so go there !😊 